# openEO process-graph layers in JupyterGIS

This notebook connects to a local [titiler-openeo](https://github.com/sentinel-hub/titiler-openeo) / openEO backend, builds a small process graph with the [`openeo`](https://cran.r-project.org/package=openeo) R client, and adds it to a `GISDocument` with `add_openeo_tile_layer()`.

Requires `jupytergis >= 0.16.0a2` (which ships the `OpenEOTileLayer` frontend). The `openeo` R client is installed on demand in the next cell; its native dependencies (`sf`/GDAL/GEOS/PROJ/udunits2) are already provided by the conda environment.

In [ ]:
# Install the openEO R client if it isn't present. The compiled dependencies
# (sf, units, s2, ...) come from the conda environment, so this only builds
# the pure-R 'openeo' package and is quick.
if (!requireNamespace("openeo", quietly = TRUE)) {
    install.packages("openeo", repos = "https://cloud.r-project.org")
}

library("jupytergis")
library("openeo")

## Connect and log in

The local dev backend listens on `http://localhost:8080` with basic auth `test` / `test`.

In [ ]:
con <- connect(host = "http://localhost:8080")
login(user = "test", password = "test", con = con)

In [ ]:
con <- connect(host = "http://localhost:8080")
login(user = "test", password = "test", con = con)

In [ ]:
# Inspect what the backend offers, then pick a collection id below.
head(list_collections(con))

## Build a process graph

Adjust `collection_id`, `bands`, and the spatial/temporal extents to match a collection your backend exposes.

In [ ]:
p <- processes()

# A globally-covered CLMS collection renders anywhere on land, unlike the
# Copernicus Contributing Missions (ccm-*), whose tasked imagery has sparse
# coverage. fapar_fapar is the Fraction of Absorbed Photosynthetically Active
# Radiation. (Backend also lists "ccm-optical", "ccm-sar", etc.)
collection_id <- "clms_fapar_global_1km_10daily_v2_cog"

cube <- p$load_collection(
    id = collection_id,
    spatial_extent = list(west = 7.5, south = 51.9, east = 7.6, north = 52.0),
    # This collection's archive ends 2020-06-30; one dekad is one timestep.
    temporal_extent = c("2020-06-01", "2020-06-10"),
    bands = c("fapar_fapar")
)

# Collapse the temporal dimension to a single image. The titiler-openeo tile
# renderer only turns a single-timestep result into a PNG; a multi-date stack
# makes save_result fail with "Expected ImageData object, got RasterStack".
# Reducing over "t" (here: the first observation) yields one renderable image.
image <- p$reduce_dimension(
    data = cube,
    dimension = "t",
    reducer = function(data, context) p$first(data)
)

result <- p$save_result(data = image, format = "PNG")

## Add the openEO layer to a JupyterGIS document

`add_openeo_tile_layer()` serializes the process graph and forwards the backend url and session bearer token, so the frontend's titiler-openeo service can render the tiles.

In [ ]:
doc <- GISDocument$new()

openeo_layer <- doc$add_openeo_tile_layer(
    result,
    connection = con,
    name = "openEO FAPAR",
    opacity = 1
)

doc